# pyINLA Interactive Demo

**pyINLA** provides fast, approximate Bayesian inference for latent Gaussian models, directly in Python.

**Why Bayesian inference with INLA?**

- **Full uncertainty quantification**: posterior distributions and credible intervals, not just point estimates
- **Fast**: results in seconds via the INLA approximation, compared to hours with MCMC sampling
- **Hierarchical models**: natural handling of grouped, spatial, and temporal data

This notebook walks through two examples (linear regression and Poisson regression), each with known true parameters so you can verify the model recovers them.

## 1. Setup

Install pyINLA and download the INLA computational binary.

In [ ]:
# Install pyinla
!pip install pyinla -q
print("pyinla installed!")

In [ ]:
# Download the INLA binary (Ubuntu 22.04 for Google Colab)
import pyinla

if not pyinla.is_binary_installed():
    print("Downloading INLA binary for Ubuntu 22.04...")
    pyinla.download_binary(os_name="Ubuntu-22.04", interactive=False)
    print("Done!")
else:
    print("INLA binary already installed")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyinla import pyinla

print("All libraries loaded!")

## 2. Bayesian Linear Regression: Exercise and Heart Rate

**Research question:** Does regular exercise lower resting heart rate?

A health researcher measures resting heart rate (bpm) and weekly exercise hours for 120 adults. A classical regression gives point estimates, but the Bayesian approach gives us the full posterior distribution of every parameter, directly answering: "How confident are we that exercise lowers heart rate, and by how much?"

We simulate data with known true parameters so we can verify the model recovers them.

| Parameter | True Value |
|---|---|
| Baseline heart rate (sedentary) | 78 bpm |
| Effect of exercise | -1.5 bpm per hour |
| Individual variation (noise SD) | 5 bpm |

In [ ]:
# Simulate data with known true parameters
np.random.seed(42)

n = 120
exercise_hours = np.random.uniform(0, 12, n)

# True parameters
TRUE_INTERCEPT = 78.0   # Resting HR for someone who doesn't exercise
TRUE_EXERCISE = -1.5    # Each hour of weekly exercise lowers HR by 1.5 bpm
TRUE_NOISE_SD = 5.0

# Generate response: heart_rate = intercept + slope * exercise + noise
heart_rate = TRUE_INTERCEPT + TRUE_EXERCISE * exercise_hours + np.random.normal(0, TRUE_NOISE_SD, n)

df = pd.DataFrame({'heart_rate': heart_rate, 'exercise': exercise_hours})

print("Sample data (first 5 rows):")
print(df.head())
print(f"\nTrue parameters: intercept = {TRUE_INTERCEPT}, exercise effect = {TRUE_EXERCISE}, noise SD = {TRUE_NOISE_SD}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['exercise'], df['heart_rate'], alpha=0.6, edgecolors='w', linewidth=0.5)

# Overlay true regression line
x_line = np.linspace(0, 12, 100)
y_line = TRUE_INTERCEPT + TRUE_EXERCISE * x_line
plt.plot(x_line, y_line, 'r--', linewidth=2, label=f'True: {TRUE_INTERCEPT} + ({TRUE_EXERCISE}) * hours')

plt.xlabel('Weekly Exercise (hours)')
plt.ylabel('Resting Heart Rate (bpm)')
plt.title('Does Exercise Lower Resting Heart Rate?')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Define and fit the Bayesian linear regression
model = {
    'response': 'heart_rate',
    'fixed': ['1', 'exercise']  # '1' = intercept
}

result = pyinla(model=model, family='gaussian', data=df)

print("Bayesian Linear Regression Results:")
print(result.summary_fixed)

# Compare true vs estimated
est_intercept = result.summary_fixed.loc['(Intercept)', 'mean']
est_exercise = result.summary_fixed.loc['exercise', 'mean']
print(f"\n{'Parameter':<20} {'True':>8} {'Estimated':>10}")
print(f"{'Intercept':<20} {TRUE_INTERCEPT:>8.1f} {est_intercept:>10.2f}")
print(f"{'Exercise effect':<20} {TRUE_EXERCISE:>8.1f} {est_exercise:>10.2f}")

In [ ]:
# Visualize posterior distributions for both parameters
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Intercept posterior
marg_int = result.marginals_fixed['(Intercept)']
axes[0].fill_between(marg_int['x'], marg_int['y'], alpha=0.3, color='steelblue')
axes[0].plot(marg_int['x'], marg_int['y'], color='steelblue', linewidth=2, label='Posterior')
axes[0].axvline(TRUE_INTERCEPT, color='red', linestyle='--', linewidth=2, label=f'True value ({TRUE_INTERCEPT})')
axes[0].set_xlabel('Intercept (bpm)')
axes[0].set_ylabel('Posterior Density')
axes[0].set_title('Baseline Heart Rate')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Exercise effect posterior
marg_ex = result.marginals_fixed['exercise']
axes[1].fill_between(marg_ex['x'], marg_ex['y'], alpha=0.3, color='steelblue')
axes[1].plot(marg_ex['x'], marg_ex['y'], color='steelblue', linewidth=2, label='Posterior')
axes[1].axvline(TRUE_EXERCISE, color='red', linestyle='--', linewidth=2, label=f'True value ({TRUE_EXERCISE})')
axes[1].axvline(0, color='gray', linestyle=':', alpha=0.5, label='No effect')
axes[1].set_xlabel('Exercise Effect (bpm per hour)')
axes[1].set_ylabel('Posterior Density')
axes[1].set_title('Effect of Exercise')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Print credible interval for the exercise effect
est = result.summary_fixed.loc['exercise', :]
print(f"Exercise effect: {est['mean']:.2f} bpm per hour")
print(f"95% credible interval: [{est['0.025quant']:.2f}, {est['0.975quant']:.2f}]")
print(f"\nThe entire 95% credible interval is below zero, confirming that")
print(f"exercise is associated with lower resting heart rate.")

In [ ]:
# Overlay the estimated regression line with 95% credible band on the data
x_pred = np.linspace(0, 12, 200)

est_int = result.summary_fixed.loc['(Intercept)', :]
est_ex = result.summary_fixed.loc['exercise', :]

y_mean = est_int['mean'] + est_ex['mean'] * x_pred
y_lower = est_int['0.025quant'] + est_ex['0.025quant'] * x_pred
y_upper = est_int['0.975quant'] + est_ex['0.975quant'] * x_pred

plt.figure(figsize=(8, 5))
plt.scatter(df['exercise'], df['heart_rate'], alpha=0.5, edgecolors='w', linewidth=0.5, label='Observed data')
plt.plot(x_pred, y_mean, color='steelblue', linewidth=2, label='Estimated regression line')
plt.fill_between(x_pred, y_lower, y_upper, alpha=0.2, color='steelblue', label='95% credible band')
plt.plot(x_line, y_line, 'r--', linewidth=1.5, alpha=0.7, label='True regression line')

plt.xlabel('Weekly Exercise (hours)')
plt.ylabel('Resting Heart Rate (bpm)')
plt.title('Bayesian Regression Fit with Uncertainty')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("The estimated line (blue) closely tracks the true line (red dashed),")
print("and the 95% credible band captures the true relationship.")

## 3. Poisson Regression: Do Speed Cameras Reduce Accidents?

**Research question:** Do speed cameras reduce traffic accidents at intersections?

A city installs speed cameras at 50 out of 100 high-risk intersections and records accident counts over one year. Since accidents are discrete counts, Poisson regression is the natural model. The Bayesian posterior tells us not just whether cameras help, but quantifies how large the effect is with full uncertainty.

**Simulated ground truth:**
- Baseline accident rate (no camera): exp(1.5) = 4.5 accidents per year
- Camera effect on log-rate: -0.4 (a 33% reduction in accidents)

In [ ]:
np.random.seed(123)

n_intersections = 100
has_camera = np.array([0] * 50 + [1] * 50)  # Balanced: 50 without, 50 with cameras

# True parameters (on the log scale)
TRUE_BASELINE = 1.5     # log(accident rate) without camera
TRUE_CAMERA = -0.4      # log-rate reduction from camera

log_rate = TRUE_BASELINE + TRUE_CAMERA * has_camera
accidents = np.random.poisson(np.exp(log_rate))

df_accidents = pd.DataFrame({
    'accidents': accidents,
    'camera': has_camera
})

print("Accident counts summary:")
print(f"  Without camera: mean = {df_accidents[df_accidents['camera']==0]['accidents'].mean():.1f} per year")
print(f"  With camera:    mean = {df_accidents[df_accidents['camera']==1]['accidents'].mean():.1f} per year")
print(f"\nTrue rates: {np.exp(TRUE_BASELINE):.1f} (no camera), {np.exp(TRUE_BASELINE + TRUE_CAMERA):.1f} (with camera)")

In [ ]:
# Fit Poisson regression
model_poisson = {
    'response': 'accidents',
    'fixed': ['1', 'camera']
}

result_poisson = pyinla(model=model_poisson, family='poisson', data=df_accidents)

print("Poisson Regression Results (log scale):")
print(result_poisson.summary_fixed)
print(f"\nTrue values: intercept = {TRUE_BASELINE}, camera = {TRUE_CAMERA}")

# Interpret on the original scale
est_baseline = result_poisson.summary_fixed.loc['(Intercept)', 'mean']
est_camera = result_poisson.summary_fixed.loc['camera', 'mean']

print(f"\nInterpretation:")
print(f"  Accident rate without camera: exp({est_baseline:.2f}) = {np.exp(est_baseline):.1f} per year")
print(f"  Accident rate with camera:    exp({est_baseline:.2f} + {est_camera:.2f}) = {np.exp(est_baseline + est_camera):.1f} per year")
print(f"  Rate ratio: exp({est_camera:.2f}) = {np.exp(est_camera):.2f}")
print(f"  Cameras are associated with a {(1 - np.exp(est_camera)) * 100:.0f}% reduction in accidents.")

## 4. Try Your Own Data

Upload a CSV file or paste data below, then fit a model.

In [ ]:
from io import StringIO

# Replace this with your own CSV data
my_csv = """
x,y
1,3.2
2,5.1
3,6.8
4,9.3
5,11.0
6,12.5
7,15.1
8,16.8
"""

my_data = pd.read_csv(StringIO(my_csv))
print("Your data:")
print(my_data)

In [ ]:
# Uncomment the lines below to upload a CSV file in Google Colab:

# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# my_data = pd.read_csv(filename)
# print(my_data.head())

In [ ]:
# Fit a model to your data
my_model = {
    'response': 'y',       # Change to your response column name
    'fixed': ['1', 'x']    # Change to your predictor column names
}

my_result = pyinla(model=my_model, family='gaussian', data=my_data)
print("Results:")
print(my_result.summary_fixed)

## 5. What's Next?

This demo covered the fundamentals. pyINLA supports much more:

**Likelihood families:**
- `gaussian`: continuous data (shown above)
- `poisson`: count data (shown above)
- `binomial`: binary or proportion data
- `gamma`, `beta`: positive or bounded continuous data
- And 15+ more families

**Random effect models:**
- `iid`: independent random effects
- `rw1`, `rw2`: random walks for smooth trends and time series
- `ar1`: autoregressive processes
- `besag`, `bym2`: spatial areal models for disease mapping
- `spde`: continuous spatial fields via stochastic PDEs

### Resources

- **Documentation**: [pyinla.org/docs](https://pyinla.org/docs)
- **Examples**: [pyinla.org/examples](https://pyinla.org/examples)
- **Applications**: [pyinla.org/apps](https://pyinla.org/apps)